# Adding memory to the agent

In this exercise, we will add short-term memory to our text-to-query agent. Short-term memory allows AI agents to have multi-turn conversations with its users.  

In this course, we will use MongoDB to persist the agent's short-term memory. We will use MongoDB's checkpointer integration with LangGraph (`MongoDBSaver`) to automatically handle the memory management for the agent.

**Run the two cells below to install the necessary libraries and enter your MongoDB connection string.**

In [33]:
!pip install -qU langgraph-checkpoint-mongodb==0.2.0 langchain-openai==0.3.28 langgraph==0.6.3  pydantic==2.11.9


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [34]:
import os
from pymongo import MongoClient

MONGODB_URI = os.environ["MONGODB_URI"]
mongodb_client = MongoClient(MONGODB_URI)

**Run the hidden cell below to re-create your agent from the end of the last chapter.**

In [35]:
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import tools_condition
from langchain_core.messages import ToolMessage
from langchain_mongodb.agent_toolkit import MONGODB_AGENT_SYSTEM_PROMPT
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langchain_mongodb.agent_toolkit.database import MongoDBDatabase
from langchain_mongodb.agent_toolkit.toolkit import MongoDBDatabaseToolkit
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import json

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Access the sample_mflix database using the MongoDBDatabase class
db = MongoDBDatabase.from_connection_string(connection_string=MONGODB_URI, database="sample_mflix")

# Initialize the MongoDB database toolkit
toolkit = MongoDBDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()
tool_map = {tool.name:tool for tool in tools}

class GraphState(TypedDict):
    messages: Annotated[list, add_messages]
# Preview the system prompt
MONGODB_AGENT_SYSTEM_PROMPT
# Create a templated prompt for the LLM
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", MONGODB_AGENT_SYSTEM_PROMPT),
        ("system", "IMPORTANT: Always start by checking your memory for relevant information before calling any tools. Do not re-run tools unless absolutely necessary. If you are not able to get enough information using the tools, reply with I DON'T KNOW. You have access to the following tools: {tool_names}."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

# Pre-fill top_k and tool_names in the prompt template
prompt = prompt.partial(top_k=5, tool_names=", ".join([tool.name for tool in tools]))

# Bind tools to the LLM
tool_augmented_llm = llm.bind_tools(tools)
# Chain the prompt and tool-augmented LLM
llm_with_tools = prompt | tool_augmented_llm

# Define the agent node
def agent_node(state):
    # Read "messages" from the graph state
    messages = state["messages"]
    # Invoke the tool-augmented LLM to determine next action
    result = llm_with_tools.invoke(messages)
    # Append the result to the graph state
    return {"messages": [result]}

# Define the tool node
def tool_node(state: GraphState):
    result = []
    # Read the tool invocation payload from the graph state
    tool_calls = state["messages"][-1].tool_calls
    for tool_call in tool_calls:
        # Extract the name of the tool to execute
        tool = tool_map[tool_call["name"]]
        # Invoke the tool with the arguments extracted from the payload
        observation = tool.invoke(tool_call["args"])
        # Create a `ToolMessage` to log the result of the tool execution
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    # Append the result to the graph state
    return {"messages": result}

# Initialize the graph with graph state
graph = StateGraph(GraphState)
# Add a node called "agent" that calls the `agent_node` function
graph.add_node("agent", agent_node)
# Add a node called "tools" that calls the `tool_node` function
graph.add_node("tools", tool_node)

# Add a fixed edge from the START node to the "agent" node
graph.add_edge(START, "agent")
# Add a fixed edge from the "tool" node to the "agent" node
graph.add_edge("tools", "agent")
# Add conditional edges from the "agent" node to the "tool" and "END" nodes
graph.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "tools", END: END},
)

**Initialize a MongoDBSaver object using the `mongodb_client` and compile the graph using it.**

In [36]:
from langgraph.checkpoint.mongodb import MongoDBSaver

checkpointer = MongoDBSaver(mongodb_client)

app = graph.compile(checkpointer=checkpointer)

The agent execution function looks similar to what we had before, except it takes an additional argument, `thread_id`.

The `thread_id` is used by the MongoDB checkpointer to retrieve the latest checkpoint for the current conversational thread upon each invocation, and write new checkpoints to MongoDB after each node execution.

To configure the thread ID, you need to create a runtime config specifying the `thread_id` and pass it as an argument while invoking the graph.

**Using the `execute_graph_with_memory()` helper function and the `thread_id` provided, find which states have the most theaters.**

In [37]:
def execute_graph_with_memory(thread_id: str, user_input: str) -> None:
  """
    Execute the memory-augmented agent

    Args:
        thread_id (str): Thread ID for which to retrieve memory
        user_input (str): User query
    """
  # Configure the thread ID
  config = {"configurable": {"thread_id": thread_id}}
  # Stream outputs from each step in the graph
  for step in app.stream(
      {"messages": [{"role": "user", "content": user_input}]},
      config,
      stream_mode="values",
  ):
      # Print the latest message from the step
      step["messages"][-1].pretty_print()

Note: due to the non-deterministic behavior of LLMs, you may sometimes find that the agent decides to make tool calls even when the information is available in its checkpoint.

In [38]:
from uuid import uuid4

thread_id = str(uuid4())
execute_graph_with_memory(thread_id, "Which states have the most theaters?")
#print(thread_id)

================================ Human Message =================================

Which states have the most theaters?
================================== Ai Message ==================================
Tool Calls:
  mongodb_list_collections (call_ZjvBsPElAHXtzdvtAow7bpcX)
 Call ID: call_ZjvBsPElAHXtzdvtAow7bpcX
  Args:
================================= Tool Message =================================

comments, embedded_movies, movies, sessions, theaters, users
================================== Ai Message ==================================
Tool Calls:
  mongodb_schema (call_oTIRtkPMGIMKOxWjGINdjQJX)
 Call ID: call_oTIRtkPMGIMKOxWjGINdjQJX
  Args:
    collection_names: theaters
================================= Tool Message =================================

Database name: sample_mflix
Collection name: theaters
Schema from a sample of documents from the collection:
_id: ObjectId
theaterId: Number
location.address.street1: String
location.address.city: String
location.address.state: String


In [39]:
execute_graph_with_memory(thread_id, "How many theaters does California have?")

================================ Human Message =================================

How many theaters does California have?
================================== Ai Message ==================================
Tool Calls:
  mongodb_query_checker (call_HVXDBlEH1gJhAjrJk4opDAA3)
 Call ID: call_HVXDBlEH1gJhAjrJk4opDAA3
  Args:
    query: db.theaters.aggregate([ { "$match": { "$expr": { "$eq": [ "$location.address.state", "CA" ] } } }, { "$count": "theaterCount" } ])
================================= Tool Message =================================

content='```javascript\ndb.theaters.aggregate([ { "$match": { "$expr": { "$eq": [ "$location.address.state", "CA" ] } } }, { "$count": "theaterCount" } ])\n```' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 137, 'total_tokens': 182, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_det

**Retrieve the checkpoint from the `thread_id` and run a new query.**

In [40]:
config = {"configurable": {"thread_id": thread_id}}

In [41]:
# Reset agent to a specific checkpoint
checkpointer.get(config)
app = graph.compile(checkpointer=checkpointer)

In [42]:
import pprint

# List and analyze checkpoints
# Remove the 'limit' argument, as get() does not accept it
checkpoints = list(checkpointer.get(config))
# If you want only the first 5 checkpoints, slice the list
pprint.pprint(checkpoints[:5])

['v', 'ts', 'id', 'channel_values', 'channel_versions']


Notice that the agent did not make any new tool calls but instead used information from its short-term memory to answer the question.

There is so much more you can do with checkpoints in LangGraph. Here are some ideas:
* Analyze successful agent trajectories
* Debug erroneous agent trajectories
* Reset the agent to a previous state
* Modify information in a previous state